In [11]:
from pyscipopt import Model, quicksum
from itertools import combinations
import logging

In [12]:
arquivo = r"C:\TCC\Documentos\Instancias\Instância Teste 15 jobs.txt"
#arquivo = Path(__file__).parent.parent / "Documentos" / "Instancias" / "Instância Teste 5 jobs.txt"

In [13]:
def ler_instancia(nome_arquivo):
    
    with open(nome_arquivo, "r", encoding="utf-8") as f:
        linhas = [linha.strip() for linha in f if linha.strip()]

    peso = {}
    familia_produto = {}
    capacidade = {}
    tempo_proc = {}
    maquinas_familia = {}
    familia = {}
    produtos_por_familia = {}

    inst = None
    secao = None
    produto_id = 1

    for linha in linhas:

        if "NÚMERO DE PRODUTOS" in linha:
            texto, inst = linha.split(":")

        # identificar seção
        if "NOME DO PRODUTO" in linha:
            secao = "produtos"
            continue

        elif "MÁQUINAS/CAPACIDADE" in linha:
            secao = "maquinas"
            continue

        elif "FAMÍLIA/TEMPO" in linha:
            secao = "familias"
            continue

        elif "---" in linha:
            continue

        # -------------------
        # PRODUTOS
        # -------------------
        if secao == "produtos":
            nome, p, f = linha.split("/")

            peso[produto_id] = float(p)
            familia_produto[produto_id] = int(f)

            produto_id += 1

        # -------------------
        # MÁQUINAS
        # -------------------
        elif secao == "maquinas":
            maq, cap = linha.split("/")

            capacidade[maq] = float(cap)

        # -------------------
        # FAMÍLIAS
        # -------------------
        elif secao == "familias":
            fam, tempo, maq = linha.split("/")

            fam = int(fam)
            familia[fam] = fam
            tempo_proc[fam] = int(tempo)
            maquinas_familia[fam] = maq.split(",")

    for produto, fam in familia_produto.items():

        if fam not in produtos_por_familia:
            produtos_por_familia[fam] = []

        produtos_por_familia[fam].append(produto)

    return (
        inst,
        peso,
        familia_produto,
        capacidade,
        tempo_proc,
        maquinas_familia,
        familia,
        produtos_por_familia
    )

def cria_lotes(capacidade, peso, produtos_por_familia, tempo_proc, familia_produto, maquinas_familia):
    
    T = []
    P = []
    lotes_validos = []
    A = []
    maquinas = list(capacidade.keys())
    produtos = list(peso.keys())
    familia_lote = []

    for fam, produtos_fam in produtos_por_familia.items():
    
            maior_capacidade = max(capacidade[m] for m in maquinas_familia[fam])
    
            for r in range(1, len(produtos_fam) + 1):
                for lote in combinations(produtos_fam, r):
                    if sum(peso[p] for p in lote) <= maior_capacidade:
                        lotes_validos.append(lote)
                        familia_lote.append(fam)
                        
    #Binariza :)
    for lotes in lotes_validos:
        linha = [1 if n in lotes else 0 for n in produtos]
        A.append(linha)

    #tempos
    for lote in lotes_validos:
        tempo_lote = tempo_proc[familia_produto[lote[0]]]
        T.append(tempo_lote)

    #pesos
    for lote in lotes_validos:
        peso_lote = sum(peso[p] for p in lote)
        P.append(peso_lote)

    return (
        T,
        P,
        lotes_validos,
        A,
        maquinas,
        produtos,
        familia_lote
    )

def cria_master(lotes_validos, produtos, A):

    modelo = Model("Master")

    X = [None for n in range(len(lotes_validos))]
    for n in range(len(X)):
        X[n] = modelo.addVar(f"X{n}", "binary")

    theta = modelo.addVar("theta", "continuous")

    for i in range(len(produtos)):
        modelo.addCons(quicksum(X[n] * A[n][i] for n in range(len(lotes_validos))) == 1)

    #modelo.setObjective(quicksum(X[n] for n in range(len(lotes_validos))) + theta, "minimize")
    modelo.setObjective(theta, "minimize")

    return(modelo, X, theta)

def cria_sub(maquinas, lotes_utilizados, maquinas_familia, familia_produto, capacidade, T, P, familia_lote):

    modelo = Model("Sub")

    y = {}
    for u in lotes_utilizados:
        for j in range(len(maquinas)):
            y[(u,j)] = modelo.addVar(f"Y{u},{j}", vtype="B")

    Cmax = modelo.addVar("Cmax", "continuous", lb=min(T))

    for u in lotes_utilizados:
        modelo.addCons(quicksum(y[(u,j)] for j in range(len(maquinas))) == 1)

    for u in lotes_utilizados:
        fam = familia_lote[u]
        for j, maquina in enumerate(maquinas):
            if maquina not in maquinas_familia[fam]:
                modelo.addCons(y[(u,j)] == 0)

    for j in range(len(maquinas)):
        modelo.addCons(quicksum(y[(u, j)] * T[u] for u in lotes_utilizados) <= Cmax)

    for u in lotes_utilizados:
        for j in range(len(maquinas)):
            modelo.addCons(P[u] * y[(u, j)] <= capacidade[maquinas[j]])

    modelo.setObjective(Cmax, "minimize")
    
    return(modelo, y, Cmax)
    
def resolve_master(modelo, X, theta):

    modelo.optimize()
    x_vals = [int(round(modelo.getVal(X[n]))) for n in range(len(X))]
    return(modelo.getObjVal(), modelo.getVal(theta), x_vals)

def resolve_sub(modelo):

    modelo.optimize()

    return(modelo.getObjVal())


In [14]:
def main(nome_arquivo):

    inst, peso, familia_produto, capacidade, tempo_proc, maquinas_familia, familia, produtos_por_familia = ler_instancia(nome_arquivo)
    T, P, lotes_validos, A, maquinas, produtos, familia_lote = cria_lotes(capacidade, peso, produtos_por_familia, tempo_proc, familia_produto, maquinas_familia)
    master, X, theta = cria_master(lotes_validos, produtos, A)
    L = min(T)

    # logging.basicConfig(
    #     filename=f'resultado{inst}produtos.log',      # Name of the log file
    #     filemode='a',            # 'a' to append logs; use 'w' to overwrite each run
    #     format='%(asctime)s - %(levelname)s - %(message)s', # Log structure
    #     level=logging.INFO       # Minimum level to capture (DEBUG, INFO, WARNING, ERROR, CRITICAL)
    # )

    Z, valor_theta, valor_x = resolve_master(master, X, theta)
    iteracao = 0
    
    while True:

        iteracao +=1
        U = [n for n, val in enumerate(valor_x) if val == 1]

        sub, y, Cmax = cria_sub(maquinas, U, maquinas_familia, familia_produto, capacidade, T, P, familia_lote)
        Q = resolve_sub(sub)

        print(f"\nITERACAO {iteracao}")
        print(f"Z = {Z}")
        print(f"theta = {valor_theta}")
        print(f"Q = {Q}")
        print(f"U = {U}")

        if Z >= Q - 1e-6:
            print("Resultado alcançado.")
            break

        master.freeTransform()

        master.addCons(theta >= (Q - L) * (quicksum(X[n] for n in U) - quicksum(X[n] for n in range(len(lotes_validos)) if n not in U)) - (Q - L) * (len(U) - 1) + L)

        print("Corte adicionado.")

        Z, valor_theta, valor_x = resolve_master(master, X, theta)
    
    print(f"Solução ótima: theta = {valor_theta:.2f}, Q = {Q:.2f}")
    print(f"Lotes selecionados: {[n for n, v in enumerate(valor_x) if v == 1]}")
    return Q



In [15]:
Q = main(arquivo)
print(Q)


ITERACAO 1
Z = 0.0
theta = 0.0
Q = 225.0
U = [0, 1, 2, 3, 4, 31, 32, 33, 34, 35, 36, 37, 158, 159, 160]
Corte adicionado.

ITERACAO 2
Z = 0.0
theta = 0.0
Q = 75.0
U = [30, 157, 164]
Corte adicionado.

ITERACAO 3
Z = 0.0
theta = 0.0
Q = 75.0
U = [30, 37, 150, 164]
Corte adicionado.

ITERACAO 4
Z = 0.0
theta = 0.0
Q = 120.0
U = [1, 28, 37, 150, 164]
Corte adicionado.

ITERACAO 5
Z = 0.0
theta = 0.0
Q = 105.0
U = [1, 28, 37, 150, 158, 163]
Corte adicionado.

ITERACAO 6
Z = 0.0
theta = 0.0
Q = 105.0
U = [1, 28, 58, 129, 158, 163]
Corte adicionado.

ITERACAO 7
Z = 0.0
theta = 0.0
Q = 180.0
U = [0, 2, 23, 31, 32, 33, 34, 35, 36, 37, 158, 159, 160]
Corte adicionado.

ITERACAO 8
Z = 0.0
theta = 0.0
Q = 225.0
U = [0, 1, 2, 3, 4, 31, 32, 33, 34, 35, 36, 37, 158, 163]
Corte adicionado.

ITERACAO 9
Z = 0.0
theta = 0.0
Q = 225.0
U = [0, 1, 2, 3, 4, 31, 32, 33, 34, 35, 36, 37, 164]
Corte adicionado.

ITERACAO 10
Z = 0.0
theta = 0.0
Q = 225.0
U = [0, 1, 2, 3, 4, 31, 32, 33, 34, 35, 36, 37, 159, 162]

KeyboardInterrupt: 